# LeetCode 1491 — Average Salary Excluding the Minimum and Maximum Salary bằng Python

**Link đề:** https://leetcode.com/problems/average-salary-excluding-the-minimum-and-maximum-salary/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def average(self, salary: list[int]) -> float:
        ...
```

### Mục tiêu học
1. Hiểu đề: bỏ **đúng một** lương nhỏ nhất và **đúng một** lương lớn nhất rồi tính trung bình phần còn lại.
2. Ôn các công cụ Python: `sum`, `min`, `max`, `len`, `sorted`, slicing, phép chia `/` và `//`.
3. Viết 4 lời giải: sắp xếp + cắt; `sum - min - max`; một lần duyệt tự theo dõi min/max/tổng; xóa phần tử.
4. Hiểu chuyện **số thực (float)** và câu "sai số trong `10^-5` được chấp nhận".
5. Trace, chứng minh, phân tích Time/Space, tự giải biến thể "trimmed mean" (trung bình cắt).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên **phân biệt** `salary`, `salary[i]` là lương của nhân viên thứ `i`.

Trả về **lương trung bình** của các nhân viên, **không tính** người lương thấp nhất và người lương cao nhất. Đáp án sai lệch trong khoảng `10^-5` so với giá trị thật được chấp nhận.

| `salary` | min | max | còn lại | Output |
|---|---:|---:|---|---:|
| `[4000,3000,1000,2000]` | 1000 | 4000 | `[3000, 2000]` | `(2000+3000)/2 = 2500.0` |
| `[1000,2000,3000]` | 1000 | 3000 | `[2000]` | `2000/1 = 2000.0` |

**Constraints:**
- `3 <= salary.length <= 100` → sau khi bỏ 2 người **luôn còn ít nhất 1** người, không lo chia cho 0.
- `1000 <= salary[i] <= 10^6`.
- Mọi giá trị **phân biệt** → min và max mỗi cái xuất hiện đúng một lần.

Câu hỏi đặt ra:
- Bỏ min/max xong thì chia cho mấy? (`n - 2`)
- Có cần sắp xếp không?
- Kết quả là `float` — Python chia thế nào?

In [1]:
salary = [4000, 3000, 1000, 2000]
rest = [s for s in salary if s != min(salary) and s != max(salary)]
print("min =", min(salary), "| max =", max(salary), "| còn lại =", rest)
print("trung bình =", sum(rest) / len(rest))

min = 1000 | max = 4000 | còn lại = [3000, 2000]
trung bình = 2500.0


Cell trên đã cho đáp án — nhưng nó gọi `min(salary)` và `max(salary)` **lặp lại trong mỗi vòng** (rất lãng phí), và chỉ đúng vì đề bảo các lương phân biệt. Ta sẽ làm gọn và hiểu rõ từng bước.

## 2. Nền tảng Python cần biết

### 2.1. `sum`, `min`, `max`, `len`

Đều là hàm có sẵn, mỗi hàm duyệt list **một lần** → `O(n)`.

In [2]:
a = [4000, 3000, 1000, 2000]
print("sum =", sum(a), "| min =", min(a), "| max =", max(a), "| len =", len(a))

sum = 10000 | min = 1000 | max = 4000 | len = 4


### 2.2. Phép chia `/` và `//`

- `/` là **chia thực**: luôn trả về `float`, kể cả khi chia hết (`6 / 2 = 3.0`).
- `//` là **chia lấy phần nguyên** (floor division): `7 // 2 = 3`.

Bài này cần trung bình có phần thập phân (ví dụ `2500.5`), nên dùng `/`.

In [3]:
print(6 / 2, type(6 / 2))
print(7 / 2)
print(7 // 2, type(7 // 2))
print("Trung bình của 2000 và 3001:", (2000 + 3001) / 2, "vs dùng // :", (2000 + 3001) // 2)

3.0 <class 'float'>
3.5
3 <class 'int'>
Trung bình của 2000 và 3001: 2500.5 vs dùng // : 2500


### 2.3. `sorted()` và slicing `[1:-1]`

- `sorted(a)` trả về **list mới** đã sắp xếp tăng dần (không sửa `a`).
- `a.sort()` sắp xếp **tại chỗ** và trả về `None`.
- `b[1:-1]` lấy từ chỉ số 1 tới **trước** phần tử cuối → bỏ phần tử đầu và phần tử cuối.

In [4]:
a = [4000, 3000, 1000, 2000]
b_sorted = sorted(a)
print("sorted(a) =", b_sorted, "| a vẫn là", a)
print("b_sorted[1:-1] =", b_sorted[1:-1])
print("a.sort() trả về:", a.sort(), "| a giờ là", a)

sorted(a) = [1000, 2000, 3000, 4000] | a vẫn là [4000, 3000, 1000, 2000]
b_sorted[1:-1] = [2000, 3000]
a.sort() trả về: None | a giờ là [1000, 2000, 3000, 4000]


### 2.4. Số thực (float) và sai số

Máy tính lưu `float` ở dạng nhị phân nên nhiều số thập phân **không biểu diễn chính xác** được. Đó là lý do đề nói "sai số trong `10^-5` được chấp nhận" — trình chấm so `abs(kết_quả - đáp_án) <= 1e-5`, không so bằng `==`.

Khi tự test, ta cũng nên so sánh bằng `abs(a - b) < 1e-9` hoặc `math.isclose`.

In [5]:
import math
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(abs((0.1 + 0.2) - 0.3) < 1e-9)
print(math.isclose(0.1 + 0.2, 0.3))
print("1e-5 =", 1e-5)

0.30000000000000004
False
True
True
1e-5 = 1e-05


## 3. Cách 1 — Sắp xếp rồi bỏ hai đầu

Ý tưởng tự nhiên nhất: sắp xếp → lương nhỏ nhất ở đầu, lớn nhất ở cuối → cắt bỏ hai đầu → lấy trung bình.

In [6]:
def average_sort(salary: list[int]) -> float:
    middle = sorted(salary)[1:-1]       # bỏ min (đầu) và max (cuối)
    return sum(middle) / len(middle)

print(average_sort([4000, 3000, 1000, 2000]))   # 2500.0
print(average_sort([1000, 2000, 3000]))         # 2000.0

2500.0
2000.0


**Độ phức tạp:** sắp xếp tốn `O(n log n)` thời gian, `O(n)` bộ nhớ cho list mới. Với `n <= 100` thì chẳng đáng kể, nhưng ta **không cần** thứ tự đầy đủ — chỉ cần biết min và max.

## 4. Nút thắt — sắp xếp làm thừa việc

Sắp xếp xác định vị trí của **mọi** phần tử, trong khi đề chỉ cần:
- tổng các phần tử ở giữa, và
- số lượng các phần tử ở giữa.

Mà **tổng giữa = tổng toàn bộ − min − max**, **số lượng giữa = n − 2**. Không cần biết thứ tự của các phần tử giữa!

In [7]:
salary = [4000, 3000, 1000, 2000]
total = sum(salary)
print("tổng toàn bộ =", total)
print("tổng giữa = tổng - min - max =", total, "-", min(salary), "-", max(salary), "=", total - min(salary) - max(salary))
print("kiểm tra bằng sort:", sum(sorted(salary)[1:-1]))

tổng toàn bộ = 10000
tổng giữa = tổng - min - max = 10000 - 1000 - 4000 = 5000
kiểm tra bằng sort: 5000


## 5. Cách 2 — `(sum - min - max) / (n - 2)` (lời giải chính)

Đây đúng là gợi ý của đề. Ba hàm có sẵn, mỗi hàm `O(n)` → tổng `O(n)`.

Mổ xẻ:
1. `total = sum(salary)`
2. `low, high = min(salary), max(salary)`
3. `return (total - low - high) / (len(salary) - 2)` — nhớ **ngoặc** cả tử và mẫu.

In [8]:
def average_sum_min_max(salary: list[int]) -> float:
    total = sum(salary)
    low, high = min(salary), max(salary)
    return (total - low - high) / (len(salary) - 2)

print(average_sum_min_max([4000, 3000, 1000, 2000]))
print(average_sum_min_max([1000, 2000, 3000]))

2500.0
2000.0


`low, high = min(salary), max(salary)` là **gán nhiều biến cùng lúc** (tuple unpacking): vế phải tạo cặp `(min, max)`, vế trái nhận lần lượt.

## 6. Cách 3 — Một lần duyệt, tự theo dõi min, max, tổng

Cách 2 duyệt list **3 lần** (sum, min, max). Ta có thể gom vào **một** vòng lặp — đây là kỹ năng nền tảng khi không có hàm dựng sẵn (hoặc khi dữ liệu đến dạng luồng, chỉ đọc được một lần).

Khởi tạo:
- `low = float('inf')` (dương vô cùng) — mọi số đều nhỏ hơn nó, nên số đầu tiên chắc chắn thay thế.
- `high = float('-inf')` (âm vô cùng).
- `total = 0`.

In [9]:
print(float('inf'), float('-inf'))
print(1000 < float('inf'), 10**6 > float('-inf'))

inf -inf
True True


In [10]:
def average_one_pass(salary: list[int]) -> float:
    low, high, total = float('inf'), float('-inf'), 0
    for s in salary:
        total += s
        if s < low:
            low = s
        if s > high:
            high = s
    return (total - low - high) / (len(salary) - 2)

print(average_one_pass([4000, 3000, 1000, 2000]))
print(average_one_pass([1000, 2000, 3000]))

2500.0
2000.0


Chú ý: dùng hai `if` riêng biệt (không phải `elif`) — với phần tử đầu tiên, nó vừa là min vừa là max.

## 7. TRACE từng bước — cách 3

Ví dụ `salary = [4000, 3000, 1000, 2000]`:

| bước | s | total | low | high |
|---:|---:|---:|---:|---:|
| khởi tạo | – | 0 | inf | -inf |
| 0 | 4000 | 4000 | 4000 | 4000 |
| 1 | 3000 | 7000 | 3000 | 4000 |
| 2 | 1000 | 8000 | 1000 | 4000 |
| 3 | 2000 | 10000 | 1000 | 4000 |

Kết quả: `(10000 - 1000 - 4000) / (4 - 2) = 5000 / 2 = 2500.0` ✔.

In [11]:
def trace_average(salary):
    low, high, total = float('inf'), float('-inf'), 0
    print(f"khởi tạo: total={total}, low={low}, high={high}")
    for i, s in enumerate(salary):
        total += s
        note = []
        if s < low:
            low = s; note.append("min mới")
        if s > high:
            high = s; note.append("max mới")
        print(f"bước {i}: s={s:>5} -> total={total:>6}, low={low:>5}, high={high:>5}  {', '.join(note)}")
    n = len(salary)
    ans = (total - low - high) / (n - 2)
    print(f"({total} - {low} - {high}) / ({n} - 2) = {ans}")
    return ans

trace_average([4000, 3000, 1000, 2000])

khởi tạo: total=0, low=inf, high=-inf
bước 0: s= 4000 -> total=  4000, low= 4000, high= 4000  min mới, max mới
bước 1: s= 3000 -> total=  7000, low= 3000, high= 4000  min mới
bước 2: s= 1000 -> total=  8000, low= 1000, high= 4000  min mới
bước 3: s= 2000 -> total= 10000, low= 1000, high= 4000  
(10000 - 1000 - 4000) / (4 - 2) = 2500.0


2500.0

Trace thêm ví dụ min và max nằm ở **giữa** mảng:

In [12]:
trace_average([6000, 9000, 1500, 3000, 7000])

khởi tạo: total=0, low=inf, high=-inf
bước 0: s= 6000 -> total=  6000, low= 6000, high= 6000  min mới, max mới
bước 1: s= 9000 -> total= 15000, low= 6000, high= 9000  max mới
bước 2: s= 1500 -> total= 16500, low= 1500, high= 9000  min mới
bước 3: s= 3000 -> total= 19500, low= 1500, high= 9000  
bước 4: s= 7000 -> total= 26500, low= 1500, high= 9000  
(26500 - 1500 - 9000) / (5 - 2) = 5333.333333333333


5333.333333333333

## 8. Cách 4 — Xóa min và max khỏi bản sao

Một cách "đúng chữ" của đề: xóa người lương thấp nhất và cao nhất rồi tính trung bình. `list.remove(x)` xóa **lần xuất hiện đầu tiên** của `x` (mất `O(n)` vì phải dời các phần tử phía sau).

Nhớ làm trên **bản sao** để không sửa input.

In [13]:
def average_remove(salary: list[int]) -> float:
    rest = salary[:]            # bản sao
    rest.remove(min(rest))
    rest.remove(max(rest))
    return sum(rest) / len(rest)

data = [4000, 3000, 1000, 2000]
print(average_remove(data), "| input giữ nguyên:", data)

2500.0 | input giữ nguyên: [4000, 3000, 1000, 2000]


## 9. Vì sao đúng?

Gọi `n = len(salary)`, `m = min`, `M = max`. Vì các lương **phân biệt** và `n >= 3`, nên `m ≠ M` và mỗi giá trị xuất hiện đúng một lần. Tập "nhân viên còn lại" có đúng `n - 2` người, và

```
tổng còn lại = (tổng tất cả) - m - M
```

vì ta chỉ bỏ đúng hai phần tử đó. Trung bình = tổng còn lại / số người còn lại = `(sum - m - M) / (n - 2)`. Mẫu số `n - 2 >= 1` nên không bao giờ chia cho 0.

Với cách 3, **invariant**: sau khi duyệt `k` phần tử đầu, `total` là tổng, `low` là min, `high` là max của `k` phần tử đó (khởi đầu với `k = 0`: tổng 0, min = +∞, max = −∞ — đúng với quy ước tập rỗng). Mỗi bước cập nhật giữ nguyên tính chất này.

## 10. Code nộp LeetCode

In [14]:
class Solution:
    def average(self, salary: list[int]) -> float:
        return (sum(salary) - min(salary) - max(salary)) / (len(salary) - 2)

sol = Solution()
assert abs(sol.average([4000, 3000, 1000, 2000]) - 2500.0) < 1e-5
assert abs(sol.average([1000, 2000, 3000]) - 2000.0) < 1e-5
print("PASS ví dụ đề:", sol.average([4000, 3000, 1000, 2000]), sol.average([1000, 2000, 3000]))

PASS ví dụ đề: 2500.0 2000.0


Bản một lần duyệt (khi phỏng vấn yêu cầu không dùng `min`/`max`):

```python
class Solution:
    def average(self, salary: list[int]) -> float:
        low, high, total = float('inf'), float('-inf'), 0
        for s in salary:
            total += s
            low = min(low, s)
            high = max(high, s)
        return (total - low - high) / (len(salary) - 2)
```

## 11. Phân tích Time / Space

Gọi `n = len(salary)`.

| Cách | Thời gian | Bộ nhớ phụ | Số lần duyệt |
|---|---|---|---|
| 1. Sort + cắt | `O(n log n)` | `O(n)` | sort + 1 |
| 2. `sum - min - max` | `O(n)` | `O(1)` | 3 |
| 3. Một lần duyệt | `O(n)` | `O(1)` | 1 |
| 4. `remove` trên bản sao | `O(n)` | `O(n)` | ~6 (copy, min, remove, max, remove, sum) |

Cả cách 2 và 3 đều `O(n)`; khác nhau chỉ ở hằng số. Không thể tốt hơn `O(n)` vì phải đọc mọi lương ít nhất một lần (lương chưa đọc có thể là min/max).

## 12. Kiểm thử: mẫu đề, biên, đối chiếu

In [15]:
def check(name, actual, expected):
    assert abs(actual - expected) < 1e-5, (name, actual, expected)
    print(f"PASS  {name:<30} -> {actual}")

tests = [
    ("đề 1",                         [4000, 3000, 1000, 2000], 2500.0),
    ("đề 2 (n = 3)",                 [1000, 2000, 3000],       2000.0),
    ("kết quả không nguyên",         [1000, 2000, 3001, 9000], 2500.5),
    ("min/max ở giữa",               [6000, 9000, 1500, 3000, 7000], (6000 + 3000 + 7000) / 3),
    ("giảm dần",                     [5000, 4000, 3000, 2000], 3500.0),
    ("biên giá trị",                 [1000, 10**6, 500000],    500000.0),
    ("chia lẻ vô hạn",               [1000, 1001, 1002, 1004, 9999], (1001 + 1002 + 1004) / 3),
]
for name, arr, exp in tests:
    for f in (average_sort, average_sum_min_max, average_one_pass, average_remove):
        assert abs(f(arr) - exp) < 1e-9, (f.__name__, arr)
    check(name, Solution().average(arr), exp)

PASS  đề 1                           -> 2500.0
PASS  đề 2 (n = 3)                   -> 2000.0
PASS  kết quả không nguyên           -> 2500.5
PASS  min/max ở giữa                 -> 5333.333333333333
PASS  giảm dần                       -> 3500.0
PASS  biên giá trị                   -> 500000.0
PASS  chia lẻ vô hạn                 -> 1002.3333333333334


In [16]:
# Đối chiếu ngẫu nhiên các cách (random.sample cho các giá trị PHÂN BIỆT đúng như constraints)
import random
random.seed(2026)
for _ in range(1000):
    n = random.randint(3, 100)
    arr = random.sample(range(1000, 10**6 + 1), n)
    exp = average_sort(arr)
    for f in (average_sum_min_max, average_one_pass, average_remove):
        assert math.isclose(f(arr), exp, rel_tol=0, abs_tol=1e-6), (f.__name__, arr)
print("1000 mảng ngẫu nhiên: 4 cách khớp nhau. PASS")

1000 mảng ngẫu nhiên: 4 cách khớp nhau. PASS


`random.sample(dãy, k)` chọn `k` phần tử **không lặp** từ dãy — tiện để sinh test đúng điều kiện "phân biệt".

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chia cho `n` thay vì `n - 2`
```python
return (sum(salary) - min(salary) - max(salary)) / len(salary)   # SAI
```

### Lỗi 2: Quên ngoặc
```python
return sum(salary) - min(salary) - max(salary) / (len(salary) - 2)   # SAI: chỉ max bị chia
```
Phép `/` ưu tiên cao hơn `-`.

### Lỗi 3: Dùng `//` → mất phần thập phân
```python
return (sum(salary) - min(salary) - max(salary)) // (len(salary) - 2)   # SAI với 2500.5
```

### Lỗi 4: Lọc bằng điều kiện "khác min và khác max" khi đề **cho phép trùng** (biến thể)
Nếu có hai người cùng lương thấp nhất, list comprehension `s != low` sẽ xóa **cả hai** — bỏ nhiều hơn một người. Bài này không bị vì lương phân biệt, nhưng đây là bẫy thường gặp ở biến thể.

### Lỗi 5: Khởi tạo `low = 0` trong cách một lần duyệt
Lương luôn `>= 1000 > 0` nên `low` sẽ mãi là `0` → sai. Phải dùng `float('inf')` hoặc `salary[0]`.

In [17]:
arr = [1000, 2000, 3001, 9000]
print("đúng        :", average_sum_min_max(arr))
print("lỗi 1 chia n:", (sum(arr) - min(arr) - max(arr)) / len(arr))
print("lỗi 2 ngoặc :", sum(arr) - min(arr) - max(arr) / (len(arr) - 2))
print("lỗi 3 //    :", (sum(arr) - min(arr) - max(arr)) // (len(arr) - 2))

dup = [1000, 1000, 2000, 3000, 5000]          # biến thể có lương trùng
low, high = min(dup), max(dup)
wrong = [s for s in dup if s != low and s != high]
print("lỗi 4: lọc bằng != còn", wrong, "-> bỏ mất 2 người lương 1000; đúng phải còn [1000, 2000, 3000]")

def wrong_low_zero(salary):
    low, high, total = 0, 0, 0
    for s in salary:
        total += s
        low = min(low, s); high = max(high, s)
    return (total - low - high) / (len(salary) - 2)
print("lỗi 5 low=0 :", wrong_low_zero([4000, 3000, 1000, 2000]), "| đúng: 2500.0")

đúng        : 2500.5
lỗi 1 chia n: 1250.25
lỗi 2 ngoặc : 9501.0
lỗi 3 //    : 2500
lỗi 4: lọc bằng != còn [2000, 3000] -> bỏ mất 2 người lương 1000; đúng phải còn [1000, 2000, 3000]
lỗi 5 low=0 : 3000.0 | đúng: 2500.0


## 14. Mẹo nhận dạng pattern

- "Bỏ phần tử lớn nhất / nhỏ nhất rồi tính…" → thường chỉ cần **tổng − min − max**, không cần sort.
- "Chỉ cần vài giá trị cực trị (min, max, top-2)" → **một lần duyệt** với biến theo dõi, `O(n)`.
- "Cần bỏ `k%` lớn nhất và nhỏ nhất" (k lớn) → lúc này **sort** lại hợp lý (LeetCode 1619 Mean of Array After Removing Some Elements).
- Kết quả là số thực + "chấp nhận sai số" → dùng `/`, test bằng `abs(a - b) < eps`.

## 15. Biến thể

### Biến thể A — Lương có thể trùng: vẫn bỏ đúng 1 min và 1 max
Công thức `sum - min - max` **vẫn đúng** (chỉ trừ một lần mỗi giá trị) — chính vì thế cách 2 bền vững hơn cách lọc `!=`.

In [18]:
dup = [1000, 1000, 2000, 3000, 5000]
print("sum - min - max:", (sum(dup) - min(dup) - max(dup)) / (len(dup) - 2))
print("kiểm tra sort  :", sum(sorted(dup)[1:-1]) / (len(dup) - 2))

sum - min - max: 2000.0
kiểm tra sort  : 2000.0


### Biến thể B — Trimmed mean: bỏ `k` người thấp nhất và `k` người cao nhất
Khi `k` tùy ý, sort rồi cắt `[k:n-k]` là cách gọn nhất (`O(n log n)`).

In [19]:
def trimmed_mean(a, k):
    mid = sorted(a)[k:len(a) - k]
    return sum(mid) / len(mid)

print(trimmed_mean([1, 2, 3, 4, 5, 6, 100], 1))   # bỏ 1 và 100 -> 4.0
print(trimmed_mean([1, 2, 3, 4, 5, 6, 100], 2))   # bỏ 1,2 và 6,100 -> 4.0

4.0
4.0


## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
Tính tay đáp án cho `salary = [8000, 9000, 2000, 3000, 6000, 1000]`. Trả về số đó.

In [20]:
def exercise_by_hand() -> float:
    # TODO: tổng = ?, min = ?, max = ?, n - 2 = ?
    pass

# assert abs(exercise_by_hand() - Solution().average([8000, 9000, 2000, 3000, 6000, 1000])) < 1e-9

### Bài 2 (điền chỗ trống — một lần duyệt)

In [21]:
def exercise_one_pass(salary: list[int]) -> float:
    low, high, total = float('inf'), float('-inf'), 0
    for s in salary:
        # TODO 1: cộng vào total
        # TODO 2: cập nhật low
        # TODO 3: cập nhật high
        pass
    # TODO 4: return ?
    pass

# assert abs(exercise_one_pass([4000, 3000, 1000, 2000]) - 2500.0) < 1e-9

### Bài 3 (min và max trong một vòng, không dùng `inf`)
Khởi tạo `low = high = salary[0]` rồi duyệt từ phần tử thứ 2. Trả về tuple `(low, high)`.

In [22]:
def exercise_min_max(a: list[int]) -> tuple[int, int]:
    # TODO
    pass

# assert exercise_min_max([4000, 3000, 1000, 2000]) == (1000, 4000)
# assert exercise_min_max([7]) == (7, 7)

### Bài 4 (LeetCode 1619 — bỏ 5% hai đầu)
Cho `arr` có độ dài là bội của 20. Bỏ 5% phần tử nhỏ nhất và 5% lớn nhất, trả về trung bình phần còn lại.

In [23]:
def exercise_trim_5_percent(arr: list[int]) -> float:
    # TODO
    pass

# assert abs(exercise_trim_5_percent([1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3]) - 2.0) < 1e-9

### Bài 5 (lương trùng — chỉ bỏ 1 min, 1 max)
Viết hàm cho biến thể A **không dùng sort**, kể cả khi có lương trùng (giả sử `len >= 3`).

In [24]:
def exercise_average_with_duplicates(salary: list[int]) -> float:
    # TODO
    pass

# assert abs(exercise_average_with_duplicates([1000, 1000, 2000, 3000, 5000]) - 2000.0) < 1e-9
# assert abs(exercise_average_with_duplicates([5, 5, 5]) - 5.0) < 1e-9

### Bài 6 (mini challenge — bỏ 2 lớn nhất & 2 nhỏ nhất trong `O(n)`)
Không sort, một lần duyệt, theo dõi **hai** giá trị nhỏ nhất và **hai** giá trị lớn nhất. Giả sử `len >= 5`, giá trị phân biệt.

In [25]:
def exercise_trim_two(salary: list[int]) -> float:
    # TODO: min1 <= min2, max1 >= max2
    pass

# assert abs(exercise_trim_two([1, 2, 3, 4, 5, 6, 7]) - 4.0) < 1e-9
# assert abs(exercise_trim_two([10, 50, 20, 40, 30]) - 30.0) < 1e-9

### Gợi ý ba mức

**Bài 1** — M1: tìm tổng, min, max. M2: tổng = 29000, min = 1000, max = 9000. M3: `(29000 - 1000 - 9000) / 4`.

**Bài 2** — M1: ba cập nhật độc lập. M2: `total += s`; `if s < low: low = s`; `if s > high: high = s`. M3: sau vòng lặp `return (total - low - high) / (len(salary) - 2)`.

**Bài 3** — M1: phần tử đầu vừa là min vừa là max. M2: `for x in a[1:]:`. M3: cập nhật bằng `if x < low` / `if x > high`, trả về `(low, high)`.

**Bài 4** — M1: sort rồi cắt. M2: `k = len(arr) // 20`. M3: `mid = sorted(arr)[k:len(arr) - k]`.

**Bài 5** — M1: cách 2 đã đúng với trùng lặp. M2: không lọc bằng `!=`. M3: `(sum - min - max) / (n - 2)`.

**Bài 6** — M1: giống "top-2" — khi gặp số nhỏ hơn `min1`, đẩy `min1` xuống `min2`. M2: `if s < min1: min2, min1 = min1, s` / `elif s < min2: min2 = s`; tương tự cho max. M3: trả về `(total - min1 - min2 - max1 - max2) / (n - 4)`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [26]:
def answer_by_hand():
    return (29000 - 1000 - 9000) / 4

def answer_one_pass(salary):
    low, high, total = float('inf'), float('-inf'), 0
    for s in salary:
        total += s
        if s < low:
            low = s
        if s > high:
            high = s
    return (total - low - high) / (len(salary) - 2)

def answer_min_max(a):
    low = high = a[0]
    for x in a[1:]:
        if x < low:
            low = x
        if x > high:
            high = x
    return (low, high)

def answer_trim_5_percent(arr):
    k = len(arr) // 20
    mid = sorted(arr)[k:len(arr) - k]
    return sum(mid) / len(mid)

def answer_average_with_duplicates(salary):
    return (sum(salary) - min(salary) - max(salary)) / (len(salary) - 2)

def answer_trim_two(salary):
    inf = float('inf')
    min1 = min2 = inf
    max1 = max2 = -inf
    total = 0
    for s in salary:
        total += s
        if s < min1:
            min2, min1 = min1, s
        elif s < min2:
            min2 = s
        if s > max1:
            max2, max1 = max1, s
        elif s > max2:
            max2 = s
    return (total - min1 - min2 - max1 - max2) / (len(salary) - 4)

assert abs(answer_by_hand() - Solution().average([8000, 9000, 2000, 3000, 6000, 1000])) < 1e-9
assert abs(answer_one_pass([4000, 3000, 1000, 2000]) - 2500.0) < 1e-9
assert answer_min_max([4000, 3000, 1000, 2000]) == (1000, 4000)
assert answer_min_max([7]) == (7, 7)
assert abs(answer_trim_5_percent([1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3]) - 2.0) < 1e-9
assert abs(answer_average_with_duplicates([1000, 1000, 2000, 3000, 5000]) - 2000.0) < 1e-9
assert abs(answer_average_with_duplicates([5, 5, 5]) - 5.0) < 1e-9
assert abs(answer_trim_two([1, 2, 3, 4, 5, 6, 7]) - 4.0) < 1e-9
assert abs(answer_trim_two([10, 50, 20, 40, 30]) - 30.0) < 1e-9
random.seed(9)
for _ in range(500):
    arr = random.sample(range(1, 10**4), random.randint(5, 40))
    assert math.isclose(answer_trim_two(arr), trimmed_mean(arr, 2), abs_tol=1e-9)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def average(self, salary: list[int]) -> float:
        return (sum(salary) - min(salary) - max(salary)) / (len(salary) - 2)
```

Template "một lần duyệt theo dõi cực trị":
```python
low, high, total = float('inf'), float('-inf'), 0
for x in a:
    total += x
    if x < low:  low = x
    if x > high: high = x
```

**Checklist trước khi submit:**
- [ ] Chia cho `n - 2`, không phải `n`?
- [ ] Có ngoặc quanh tử số?
- [ ] Dùng `/` (chia thực), không phải `//`?
- [ ] Nếu tự theo dõi min/max: khởi tạo bằng `inf` hoặc phần tử đầu, không phải `0`?
- [ ] Test bằng sai số (`abs(a - b) < 1e-5`), không so `==` với float?